# Lecture 2: Value Prediction Problems
This notebook addresses the problem of estimating the value function of a Markov Reward Process (MRP) through interaction, without access to the model.

---
**Topics:**
1. Tabular TD(0) — Temporal Difference Learning
2. Every-Visit Monte Carlo
3. TD(λ) — Unifying Monte Carlo and TD
4. Linear Function Approximation for Large State Spaces
5. Least-Squares TD (LSTD)
6. Comprehensive Comparison


## 2.1 Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

# ── Markov Reward Process ─────────────────────────────────────────
class MRP:
    """Markov Reward Process — used for value prediction."""
    def __init__(self, states, transitions, rewards, gamma=0.95):
        self.states = states
        self.n_states = len(states)
        self.state_idx = {s: i for i, s in enumerate(states)}
        self.transitions = transitions   # {s: {s': prob}}
        self.rewards = rewards           # {s: float}
        self.gamma = gamma

    def step(self, state):
        """Simulate one transition -> (next_state, reward)."""
        next_states = list(self.transitions[state].keys())
        probs       = list(self.transitions[state].values())
        next_state  = np.random.choice(next_states, p=probs)
        reward      = self.rewards.get(state, 0.0)
        return next_state, reward

    def true_value(self):
        """Solve the Bellman linear system directly: V = (I - gamma*P)^{-1} r."""
        n = self.n_states
        R = np.array([self.rewards.get(s, 0.0) for s in self.states])
        P = np.zeros((n, n))
        for i, s in enumerate(self.states):
            for s_next, prob in self.transitions.get(s, {}).items():
                j = self.state_idx[s_next]
                P[i, j] = prob
        return np.linalg.solve(np.eye(n) - self.gamma * P, R)

def create_random_walk(n_states=7, gamma=0.99):
    """
    Classic Random Walk MRP (adapted from Sutton & Barto).
    Terminal states at both ends; reward +1 only when stepping right
    from the penultimate state.
    """
    states = list(range(n_states))
    transitions, rewards = {}, {}
    for s in states:
        if s == 0 or s == n_states - 1:
            transitions[s] = {s: 1.0}
            rewards[s]     = 0.0
        else:
            transitions[s] = {s - 1: 0.5, s + 1: 0.5}
            rewards[s]     = 1.0 if s == n_states - 2 else 0.0
    return MRP(states, transitions, rewards, gamma)

mrp    = create_random_walk(n_states=7)
V_true = mrp.true_value()
print("Random Walk MRP created.")
print(f"True value function V: {np.round(V_true, 3)}")

## 2.2 Tabular TD(0)

### Algorithm 1 (Szepesvári, p. 12)

```
function TD0(X, R, Y, V):
    δ ← R + γ·V[Y] − V[X]
    V[X] ← V[X] + α·δ
    return V
```

**Temporal difference error:**
$$\delta_{t+1} = R_{t+1} + \gamma \hat{V}_t(X_{t+1}) - \hat{V}_t(X_t)$$

**Update rule:**
$$\hat{V}_{t+1}(x) = \hat{V}_t(x) + \alpha_t \, \delta_{t+1} \, \mathbf{1}[X_t = x]$$

**Bootstrapping:** TD(0) uses the next-step estimate as a target — the key distinction from Monte Carlo.

**Convergence guarantee:** Under the Robbins-Monro conditions ($\sum \alpha_t = \infty$, $\sum \alpha_t^2 < \infty$), $\hat{V}_t \to V$ almost surely (Borkar 1998).


In [ ]:
def td0(mrp, n_steps=50_000, alpha=0.05, alpha_decay=False):
    """
    Tabular TD(0) — Algorithm 1 (Szepesvari)

    Step size:
      constant : alpha_t = alpha
      decaying : alpha_t = alpha / (1 + t^0.6)  [satisfies Robbins-Monro]
    """
    V     = np.zeros(mrp.n_states)
    errors = []
    state  = mrp.states[len(mrp.states) // 2]

    for t in range(n_steps):
        next_state, reward = mrp.step(state)
        alpha_t = alpha / (1 + t ** 0.6) if alpha_decay else alpha

        i = mrp.state_idx[state]
        j = mrp.state_idx[next_state]
        td_error = reward + mrp.gamma * V[j] - V[i]
        V[i] += alpha_t * td_error

        if t % 500 == 0:
            errors.append(np.sqrt(np.mean((V - V_true) ** 2)))

        if next_state in [0, mrp.n_states - 1]:
            state = mrp.states[len(mrp.states) // 2]
        else:
            state = next_state

    return V, errors

alphas = [0.01, 0.05, 0.1, 0.3]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
all_V, all_err = {}, {}

for alpha in alphas:
    V_est, errs = td0(mrp, n_steps=30_000, alpha=alpha)
    all_V[alpha]   = V_est
    all_err[alpha] = errs

x_pos = np.arange(mrp.n_states)
axes[0].plot(x_pos, V_true, 'k--', linewidth=3, label='True V', zorder=5)
for alpha in alphas:
    axes[0].plot(x_pos, all_V[alpha], '-o', markersize=6, linewidth=2, label=f'\u03b1={alpha}')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f's{i}' for i in mrp.states])
axes[0].set_xlabel('State')
axes[0].set_ylabel('Value Estimate')
axes[0].set_title('TD(0): Different Learning Rates (30K steps)', fontweight='bold')
axes[0].legend()

step_ticks = np.arange(len(all_err[0.05])) * 500
for alpha in alphas:
    axes[1].plot(step_ticks, all_err[alpha], linewidth=2, label=f'\u03b1={alpha}')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('RMSE')
axes[1].set_title('TD(0) Convergence (RMSE)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_td0_en.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.3 Every-Visit Monte Carlo

### Algorithm 2 (Szepesvári, p. 15)

For an episodic MRP, compute returns backwards at the end of each episode:

```
function EveryVisitMC(X₀, R₁, X₁, ..., X_{T-1}, R_T, V):
    sum ← 0
    for t = T-1 downto 0:
        sum ← R_{t+1} + γ·sum        (efficient backward accumulation)
        V[Xₜ] ← V[Xₜ] + α·(sum − V[Xₜ])
    return V
```

**Return:** $R_t = \sum_{s=t}^{T-1} \gamma^{s-t} R_{s+1}$ (multi-step prediction)

**Key distinction:** Monte Carlo does **not** bootstrap — it uses full returns.


In [ ]:
def generate_episode(mrp, start_state=None, max_steps=500):
    """Generate one episode: returns (states, rewards) lists."""
    if start_state is None:
        start_state = mrp.states[len(mrp.states) // 2]
    states_ep, rewards_ep = [start_state], []
    state = start_state
    for _ in range(max_steps):
        next_state, reward = mrp.step(state)
        rewards_ep.append(reward)
        states_ep.append(next_state)
        state = next_state
        if next_state in [0, mrp.n_states - 1]:
            break
    return states_ep[:-1], rewards_ep

def every_visit_mc(mrp, n_episodes=5_000, alpha=0.05):
    """Every-Visit Monte Carlo — Algorithm 2 (Szepesvari)."""
    V = np.zeros(mrp.n_states)
    errors = []
    for ep in range(n_episodes):
        states_ep, rewards_ep = generate_episode(mrp)
        G = 0.0
        for t in range(len(rewards_ep) - 1, -1, -1):
            G     = rewards_ep[t] + mrp.gamma * G
            i     = mrp.state_idx[states_ep[t]]
            V[i] += alpha * (G - V[i])
        if ep % 50 == 0:
            errors.append(np.sqrt(np.mean((V - V_true) ** 2)))
    return V, errors

V_mc, err_mc = every_visit_mc(mrp, n_episodes=5_000, alpha=0.05)
V_td, err_td = td0(mrp, n_steps=50_000, alpha=0.05)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x_pos = np.arange(mrp.n_states)
width = 0.25
axes[0].bar(x_pos - width, V_true, width, label='True V',       color='black',      alpha=0.7)
axes[0].bar(x_pos,          V_td,  width, label='TD(0)',         color='steelblue',  alpha=0.8)
axes[0].bar(x_pos + width,  V_mc,  width, label='Monte Carlo',   color='darkorange', alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f's{i}' for i in mrp.states])
axes[0].set_xlabel('State'); axes[0].set_ylabel('Value')
axes[0].set_title('TD(0) vs Monte Carlo: Value Estimates', fontweight='bold')
axes[0].legend()

ep_ticks = np.arange(len(err_mc)) * 50
axes[1].plot(ep_ticks, err_mc, 'darkorange', linewidth=2,
             label=f'MC (\u03b1=0.05, RMSE={err_mc[-1]:.4f})')
axes[1].plot(np.linspace(0, 5000, len(err_td)), err_td, 'steelblue', linewidth=2,
             label=f'TD(0) (\u03b1=0.05, RMSE={err_td[-1]:.4f})')
axes[1].set_xlabel('Episode (or step equivalent)'); axes[1].set_ylabel('RMSE')
axes[1].set_title('Convergence Comparison', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_mc_vs_td_en.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Final RMSE -> MC: {err_mc[-1]:.5f}  |  TD(0): {err_td[-1]:.5f}")

## 2.4 TD(λ) — Unifying Monte Carlo and TD

TD(λ) bridges TD(0) and Monte Carlo via **eligibility traces**.

### n-step Return
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n \hat{V}(X_{t+n})$$

- $n=1$ → TD(0)  
- $n=\infty$ → Monte Carlo

### λ-return (geometrically weighted average)
$$G_t^\lambda = (1-\lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}$$

### Eligibility Trace Update

$$z_{t+1}(x) = \gamma \lambda \, z_t(x) + \mathbf{1}[X_t = x]$$
$$\hat{V}_{t+1}(x) = \hat{V}_t(x) + \alpha_t \, \delta_{t+1} \, z_{t+1}(x)$$


In [ ]:
def td_lambda(mrp, lam=0.5, alpha=0.05, n_episodes=3_000):
    """
    TD(lambda) with eligibility traces — backward-view implementation.
    lam=0 -> TD(0),  lam=1 -> Monte Carlo equivalent.
    """
    V      = np.zeros(mrp.n_states)
    errors = []

    for ep in range(n_episodes):
        z     = np.zeros(mrp.n_states)
        state = mrp.states[len(mrp.states) // 2]

        for _ in range(500):
            next_state, reward = mrp.step(state)
            i = mrp.state_idx[state]
            j = mrp.state_idx[next_state]

            delta = reward + mrp.gamma * V[j] - V[i]
            z    *= mrp.gamma * lam
            z[i] += 1.0
            V    += alpha * delta * z

            if next_state in [0, mrp.n_states - 1]:
                z[:] = 0
                state = mrp.states[len(mrp.states) // 2]
            else:
                state = next_state

        if ep % 30 == 0:
            errors.append(np.sqrt(np.mean((V - V_true) ** 2)))

    return V, errors

lambdas = [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]
colors  = plt.cm.plasma(np.linspace(0, 0.9, len(lambdas)))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
final_rmse = []

for lam, col in zip(lambdas, colors):
    V_lam, errs_lam = td_lambda(mrp, lam=lam, alpha=0.05, n_episodes=2_000)
    axes[0].plot(np.arange(len(errs_lam)) * 30, errs_lam,
                 color=col, linewidth=2, label=f'\u03bb={lam}')
    final_rmse.append(errs_lam[-1])

axes[0].set_xlabel('Episode'); axes[0].set_ylabel('RMSE')
axes[0].set_title('TD(\u03bb): Different \u03bb Values (\u03b1=0.05)', fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].plot(lambdas, final_rmse, 'ko-', markersize=8, linewidth=2)
axes[1].fill_between(lambdas, final_rmse, alpha=0.2)
axes[1].set_xlabel('\u03bb value'); axes[1].set_ylabel('Final RMSE (after 2000 episodes)')
axes[1].set_title('TD(\u03bb): Effect of \u03bb Selection', fontweight='bold')
best_lam = lambdas[np.argmin(final_rmse)]
axes[1].axvline(x=best_lam, color='red', linestyle='--', label=f'Best \u03bb = {best_lam}')
axes[1].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_td_lambda_en.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Best lambda = {best_lam}  (RMSE = {min(final_rmse):.5f})")

## 2.5 Linear Function Approximation for Large State Spaces

When the state space is large, tabular representation becomes infeasible. We use **function approximation**:

$$\hat{V}(x; \theta) = \theta^\top \phi(x)$$

where $\phi(x) \in \mathbb{R}^d$ is a feature vector and $d \ll |\mathcal{X}|$.

### Linear TD(0) with Function Approximation (Szepesvári, p. 22)

$$\theta_{t+1} = \theta_t + \alpha_t \delta_{t+1} \phi(X_t)$$

Eligibility trace:
$$z_{t+1} = \gamma \lambda z_t + \phi(X_t)$$

TD error:
$$\delta_{t+1} = R_{t+1} + \gamma \hat{V}(X_{t+1}; \theta_t) - \hat{V}(X_t; \theta_t)$$


In [ ]:
def linear_td0_continuous(n_states_sim=200, n_features=50, n_steps=20_000,
                           alpha=0.01, gamma=0.95):
    """
    Linear TD(0) in a continuous state space.
    Simplified version of Algorithm 4 (Szepesvari, p. 22).
    """
    np.random.seed(42)
    Phi = np.random.randn(n_states_sim, n_features)
    Phi /= np.linalg.norm(Phi, axis=1, keepdims=True)

    true_weights = np.random.randn(n_features)
    V_true_cont  = Phi @ true_weights

    P = np.zeros((n_states_sim, n_states_sim))
    for i in range(n_states_sim):
        neighbors = [max(0, i-3), max(0, i-1), i,
                     min(n_states_sim-1, i+1), min(n_states_sim-1, i+3)]
        for nb in neighbors:
            P[i, nb] += 1.0
        P[i] /= P[i].sum()

    R_fn   = 0.1 * np.random.randn(n_states_sim)
    theta  = np.zeros(n_features)
    errors = []
    state  = n_states_sim // 2

    for t in range(n_steps):
        next_state = np.random.choice(n_states_sim, p=P[state])
        reward     = R_fn[state]
        phi_s      = Phi[state]
        phi_ns     = Phi[next_state]
        td_error   = reward + gamma * (theta @ phi_ns) - (theta @ phi_s)
        theta     += alpha * td_error * phi_s
        state      = next_state
        if t % 200 == 0:
            errors.append(np.sqrt(np.mean((Phi @ theta - V_true_cont) ** 2)))

    return Phi @ theta, V_true_cont, errors, theta

V_est, V_true_cont, errors_fa, theta = linear_td0_continuous()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x_axis = np.arange(len(V_true_cont))
axes[0].plot(x_axis, V_true_cont, 'k-',  linewidth=2, label='True V', alpha=0.7)
axes[0].plot(x_axis, V_est,       'r--', linewidth=2, label='TD(0) Estimate (Linear FA)')
axes[0].set_xlabel('State index'); axes[0].set_ylabel('Value')
axes[0].set_title('TD(0) with Linear Function Approximation', fontweight='bold')
axes[0].legend()

step_ax = np.arange(len(errors_fa)) * 200
axes[1].semilogy(step_ax, errors_fa, 'purple', linewidth=2)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('RMSE (log scale)')
axes[1].set_title('Linear TD(0) Convergence', fontweight='bold')
axes[1].fill_between(step_ax, errors_fa, alpha=0.2, color='purple')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_func_approx_en.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Final RMSE: {errors_fa[-1]:.5f}")

## 2.6 Least-Squares TD (LSTD)

Instead of incremental gradient updates, LSTD solves the projected Bellman equation in closed form.

### LSTD(0) — Algorithm 7 (Szepesvári, p. 29)

$$A\theta = b$$

$$A = \sum_{t} \phi(X_t)\bigl[\phi(X_t) - \gamma \phi(X_{t+1})\bigr]^\top, \qquad b = \sum_t \phi(X_t) R_{t+1}$$

**Advantage:** No iterative updates; a single solve yields the solution.  
**Cost:** $O(d^2)$ memory and $O(d^3)$ computation per batch.


In [ ]:
def lstd(Phi, n_steps=5_000, gamma=0.95):
    """
    LSTD(0) — Least-Squares Temporal Difference
    Algorithm 7 (Szepesvari, p. 29), simplified version.

    Phi : (n_states x d) feature matrix
    """
    n_states, d = Phi.shape
    A = np.zeros((d, d))
    b = np.zeros(d)
    state = n_states // 2

    P = np.zeros((n_states, n_states))
    for i in range(n_states):
        for nb in [max(0, i-1), min(n_states-1, i+1)]:
            P[i, nb] += 0.5

    R_fn = np.zeros(n_states)
    R_fn[-2] = 1.0

    for _ in range(n_steps):
        next_state = np.random.choice(n_states, p=P[state])
        reward     = R_fn[state]
        phi_s      = Phi[state]
        phi_ns     = Phi[next_state]
        A += np.outer(phi_s, phi_s - gamma * phi_ns)
        b += phi_s * reward
        state = next_state

    theta_lstd = np.linalg.solve(A + 1e-6 * np.eye(d), b)
    return theta_lstd, A, b

np.random.seed(0)
n_s, d = 50, 10
Phi_test    = np.random.randn(n_s, d)
Phi_test   /= np.linalg.norm(Phi_test, axis=1, keepdims=True)
theta_true  = np.random.randn(d)
V_true_test = Phi_test @ theta_true

theta_lstd, _, _ = lstd(Phi_test, n_steps=10_000)
V_lstd = Phi_test @ theta_lstd

fig, ax = plt.subplots(figsize=(12, 5))
x_ax = np.arange(n_s)
ax.plot(x_ax, V_true_test, 'k-',  linewidth=3, label='True V', zorder=5)
ax.plot(x_ax, V_lstd,      'r--', linewidth=2,
        label=f'LSTD (RMSE={np.sqrt(np.mean((V_lstd-V_true_test)**2)):.4f})')
ax.set_xlabel('State index'); ax.set_ylabel('Value')
ax.set_title('LSTD(0): Least-Squares Temporal Difference', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_lstd_en.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"LSTD RMSE: {np.sqrt(np.mean((V_lstd - V_true_test)**2)):.5f}")

## 2.7 Comprehensive Comparison

In [ ]:
mrp_cmp    = create_random_walk(n_states=7, gamma=0.95)
V_true_cmp = mrp_cmp.true_value()

results = {}
for alpha_val in [0.02, 0.05, 0.1]:
    V_td0, _ = td0(mrp_cmp, n_steps=20_000, alpha=alpha_val)
    V_mc_a,_ = every_visit_mc(mrp_cmp, n_episodes=2_000, alpha=alpha_val)
    V_td5, _ = td_lambda(mrp_cmp, lam=0.5, alpha=alpha_val, n_episodes=2_000)
    results[alpha_val] = {
        'TD(0)':   np.sqrt(np.mean((V_td0 - V_true_cmp)**2)),
        'MC':      np.sqrt(np.mean((V_mc_a - V_true_cmp)**2)),
        'TD(0.5)': np.sqrt(np.mean((V_td5 - V_true_cmp)**2)),
    }

fig, ax = plt.subplots(figsize=(11, 5))
methods     = ['TD(0)', 'MC', 'TD(0.5)']
alphas_plot = [0.02, 0.05, 0.1]
x = np.arange(len(methods))
width = 0.25
colors_bar = ['steelblue', 'darkorange', 'green']

for i, (alpha_val, col) in enumerate(zip(alphas_plot, colors_bar)):
    rmses = [results[alpha_val][m] for m in methods]
    ax.bar(x + i*width, rmses, width, label=f'\u03b1={alpha_val}', color=col, alpha=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels(methods, fontsize=12)
ax.set_ylabel('Final RMSE')
ax.set_title('Value Prediction Algorithms Comparison\n(20K steps / 2K episodes)', fontweight='bold')
ax.legend()

best_method = min(methods, key=lambda m: results[0.05][m])
print(f"Best method (\u03b1=0.05): {best_method}")

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_comparison_en.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n--- Results Table ---")
print(f"{'Method':<15} {'\u03b1=0.02':>10} {'\u03b1=0.05':>10} {'\u03b1=0.10':>10}")
print("-" * 50)
for m in methods:
    row = f"{m:<15}" + "".join(f"{results[a][m]:>10.5f}" for a in alphas_plot)
    print(row)

## 2.8 Summary

| Algorithm | Bootstrapping | Terminal Required | Convergence | Variance |
|-----------|:-------------:|:-----------------:|-------------|:--------:|
| **TD(0)**       | ✅ Yes | ❌ No  | Almost sure (RM conditions) | Low  |
| **Monte Carlo** | ❌ No  | ✅ Yes | Almost sure (RM conditions) | High |
| **TD(λ)**       | ✅ Partial | ✅/❌ depends on λ | Almost sure | Medium |
| **LSTD(0)**     | ✅ Yes | ❌ No  | Single batch solve | Very low |

### When to Use Which?
- **TD(0):** Online learning; state space of manageable size.
- **Monte Carlo:** Episodic problems; when bootstrapping bias is undesirable.
- **TD(λ):** Balance between bias and variance; λ is a tuneable hyperparameter.
- **LSTD:** Batch (offline) scenarios with high data efficiency requirements.
